# Machine Learning Lab – Logistic Regression

**Roll No.: CH.SC.U4CSE24257**

In [1]:
# Import required libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import statsmodels.api as sm
import statsmodels.formula.api as smf

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    roc_auc_score
)

import warnings
warnings.filterwarnings("ignore")

ModuleNotFoundError: No module named 'statsmodels'

In [ ]:
# Load UCLA binary logistic regression dataset

url = "https://stats.idre.ucla.edu/stat/stata/dae/binary.dta"

df = pd.read_stata(url)

print("Dataset loaded successfully!")
print("Shape:", df.shape)

df.head()

In [ ]:
# Display information about the dataset

print("Dataset Information:")
print("--------------------")

df.info()

In [ ]:
# Check dimensions

print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

# Display column names

print("\nColumn names:")
print(df.columns.tolist())

In [ ]:
# Check missing values

print("Missing values in each column:")
print(df.isnull().sum())

In [ ]:
# Check duplicate records

print("Number of duplicate rows:", df.duplicated().sum())

In [ ]:
# Descriptive statistics

df.describe()

In [ ]:
print("Mean GRE:", df['gre'].mean())
print("Mean GPA:", df['gpa'].mean())
print("Mean Rank:", df['rank'].mean())
print("Admission Rate:", df['admit'].mean())

In [ ]:
# Frequency of admission outcomes

print(df['admit'].value_counts())

print("\nAdmission proportions:")
print(df['admit'].value_counts(normalize=True))

In [ ]:
plt.figure(figsize=(6,4))

sns.countplot(x='admit', data=df)

plt.title("Admission Status Distribution")
plt.xlabel("Admission (0 = Not Admitted, 1 = Admitted)")
plt.ylabel("Number of Students")

plt.show()

In [ ]:
print("Number of students from each institution rank:")
print(df['rank'].value_counts().sort_index())

In [ ]:
plt.figure(figsize=(7,4))

sns.countplot(x='rank', data=df)

plt.title("Distribution of Undergraduate Institution Rank")
plt.xlabel("Institution Rank")
plt.ylabel("Number of Students")

plt.show()

In [ ]:
# Cross-tabulation

rank_admit = pd.crosstab(df['rank'], df['admit'])

print(rank_admit)

In [ ]:
admission_by_rank = df.groupby('rank')['admit'].mean() * 100

print("Admission percentage by institution rank:")
print(admission_by_rank)

In [ ]:
plt.figure(figsize=(7,4))

sns.barplot(
    x=admission_by_rank.index,
    y=admission_by_rank.values
)

plt.title("Admission Rate by Institution Rank")
plt.xlabel("Institution Rank")
plt.ylabel("Admission Rate (%)")

plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(df['gre'], kde=True, ax=axes[0])
axes[0].set_title("Distribution of GRE Scores")
axes[0].set_xlabel("GRE")

sns.histplot(df['gpa'], kde=True, ax=axes[1])
axes[1].set_title("Distribution of GPA")
axes[1].set_xlabel("GPA")

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7,5))

sns.boxplot(x='admit', y='gre', data=df)

plt.title("GRE Scores by Admission Status")
plt.xlabel("Admission (0 = No, 1 = Yes)")
plt.ylabel("GRE Score")

plt.show()

In [ ]:
plt.figure(figsize=(7,5))

sns.boxplot(x='admit', y='gpa', data=df)

plt.title("GPA by Admission Status")
plt.xlabel("Admission (0 = No, 1 = Yes)")
plt.ylabel("GPA")

plt.show()

In [ ]:
plt.figure(figsize=(7,5))

corr = df[['admit', 'gre', 'gpa', 'rank']].corr()

sns.heatmap(
    corr,
    annot=True,
    cmap='coolwarm',
    fmt='.2f'
)

plt.title("Correlation Matrix")

plt.show()

In [ ]:
# Logistic Regression Model
# C(rank) treats institution rank as a categorical variable

model = smf.logit(
    'admit ~ gre + gpa + C(rank)',
    data=df
).fit()

print(model.summary())

In [ ]:
# Calculate odds ratios

odds_ratios = np.exp(model.params)

print("Odds Ratios:")
print(odds_ratios)

In [ ]:
# Odds ratio table

odds_table = pd.DataFrame({
    'Coefficient': model.params,
    'Odds Ratio': np.exp(model.params),
    'P-value': model.pvalues,
    'Lower CI': np.exp(model.conf_int()[0]),
    'Upper CI': np.exp(model.conf_int()[1])
})

odds_table

In [ ]:
# Display p-values

print("P-values:")
print(model.pvalues)

In [ ]:
# Null model with intercept only

null_model = smf.logit(
    'admit ~ 1',
    data=df
).fit(disp=0)

# Likelihood ratio test

lr_stat = 2 * (model.llf - null_model.llf)
df_lr = model.df_model
p_value_lr = model.llr_pvalue

print("Likelihood Ratio Statistic:", lr_stat)
print("Degrees of Freedom:", df_lr)
print("P-value:", p_value_lr)

In [ ]:
print("McFadden's Pseudo R²:", model.prsquared)

In [ ]:
df['predicted_probability'] = model.predict(df)

df[['admit', 'gre', 'gpa', 'rank', 'predicted_probability']].head(10)

In [ ]:
df['predicted_class'] = (
    df['predicted_probability'] >= 0.5
).astype(int)

df[['admit', 'predicted_probability', 'predicted_class']].head(10)

In [ ]:
cm = confusion_matrix(
    df['admit'],
    df['predicted_class']
)

print("Confusion Matrix:")
print(cm)

In [ ]:
plt.figure(figsize=(6,5))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues'
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")

plt.show()

In [ ]:
accuracy = accuracy_score(
    df['admit'],
    df['predicted_class']
)

print("Accuracy:", accuracy)
print("Accuracy (%):", accuracy * 100)

In [ ]:
print(
    classification_report(
        df['admit'],
        df['predicted_class']
    )
)

In [ ]:
# Calculate ROC curve

fpr, tpr, thresholds = roc_curve(
    df['admit'],
    df['predicted_probability']
)

auc = roc_auc_score(
    df['admit'],
    df['predicted_probability']
)

print("AUC:", auc)

In [ ]:
plt.figure(figsize=(7,5))

plt.plot(
    fpr,
    tpr,
    label=f'Logistic Regression (AUC = {auc:.3f})'
)

plt.plot(
    [0, 1],
    [0, 1],
    linestyle='--'
)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")

plt.title("ROC Curve")

plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(8,5))

sns.histplot(
    data=df,
    x='predicted_probability',
    hue='admit',
    bins=20,
    kde=True
)

plt.title("Predicted Admission Probabilities")
plt.xlabel("Predicted Probability of Admission")
plt.ylabel("Count")

plt.show()

In [ ]:
new_student = pd.DataFrame({
    'gre': [700],
    'gpa': [3.8],
    'rank': [1]
})

probability = model.predict(new_student)[0]

print("Predicted probability of admission:", probability)

if probability >= 0.5:
    print("Predicted outcome: ADMITTED")
else:
    print("Predicted outcome: NOT ADMITTED")

In [ ]:
new_student = pd.DataFrame({
    'gre': [550],
    'gpa': [3.2],
    'rank': [4]
})

probability = model.predict(new_student)[0]

print("Predicted probability:", probability)

In [ ]:
print("Logistic Regression Coefficients:")
print(model.params)

## Conclusion

In [ ]:
print("Logistic Regression experiment completed successfully.")